In [1]:
# Cell 1: install libraries, check GPU, set all settings in one place, print versions
!pip install -q transformers accelerate

import os, random, sys
import numpy as np, pandas as pd, torch, sklearn, transformers

# All settings live here (change them only here)
CONFIG = {
    "seed": 42,
    "model_name": "distilbert-base-uncased",
    "max_length": 128,
    "batch_size": 32,
    "learning_rate": 2e-5,
    "epochs": 2,
    "thresholds": [round(t, 2) for t in np.arange(0.1, 0.91, 0.1)],   # thresholds to try on validation
    "split": {"train": 0.70, "val": 0.15, "test": 0.15},
    "label_columns": ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"],
    "work_dir": "/content/moderation",                      # data and checkpoints
    "output_dir": "/content/moderation/deployable",         # final files for download
}
os.makedirs(CONFIG["work_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Same seed everywhere so results can be repeated
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

# GPU check and versions
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("NO GPU: in Colab go to Runtime > Change runtime type > T4 GPU, then run this cell again")
print("\nVersions -> python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| transformers:", transformers.__version__, "| scikit-learn:", sklearn.__version__,
      "| pandas:", pd.__version__, "| numpy:", np.__version__)
print("Settings saved in CONFIG:", list(CONFIG.keys()))

GPU available: True
GPU name: Tesla T4

Versions -> python: 3.13.15 | torch: 2.11.0+cu130 | transformers: 5.18.0 | scikit-learn: 1.6.1 | pandas: 2.2.3 | numpy: 2.1.3
Settings saved in CONFIG: ['seed', 'model_name', 'max_length', 'batch_size', 'learning_rate', 'epochs', 'thresholds', 'split', 'label_columns', 'work_dir', 'output_dir']


In [2]:
# Cell 2: upload train.csv (or train.csv.zip), load it, and look at the data
import zipfile
from google.colab import files

uploaded = files.upload()                       # choose train.csv or train.csv.zip

# Save the uploaded file(s) in the work folder; unzip if needed
for name, content in uploaded.items():
    path = os.path.join(CONFIG["work_dir"], name)
    with open(path, "wb") as f:
        f.write(content)
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(CONFIG["work_dir"])
print("Files in work folder:", os.listdir(CONFIG["work_dir"]))

# Load the training file
df = pd.read_csv(os.path.join(CONFIG["work_dir"], "train.csv"))
print("\nShape (rows, columns):", df.shape)
print("Columns:", list(df.columns))

# Label counts and the share of comments with at least one label
labels = CONFIG["label_columns"]
print("\nLabel counts (comments with label = 1):")
print(df[labels].sum().to_string())
any_flag = (df[labels].sum(axis=1) > 0)
print(f"\nComments with at least one label: {any_flag.sum():,} ({any_flag.mean():.2%})")

# Missing text and text length summary (in characters and words)
print("\nMissing comment_text:", df["comment_text"].isna().sum())
lengths = df["comment_text"].fillna("").str.len()
words = df["comment_text"].fillna("").str.split().str.len()
print("\nText length in characters:")
print(lengths.describe().round(1).to_string())
print("\nText length in words (median / 95th percentile):", words.median(), "/", words.quantile(0.95))

Saving train.csv to train.csv
Files in work folder: ['train.csv', 'deployable']

Shape (rows, columns): (159571, 8)
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Label counts (comments with label = 1):
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405

Comments with at least one label: 16,225 (10.17%)

Missing comment_text: 0

Text length in characters:
count    159571.0
mean        394.1
std         590.7
min           6.0
25%          96.0
50%         205.0
75%         435.0
max        5000.0

Text length in words (median / 95th percentile): 36.0 / 230.0


In [3]:
# Cell 3: clean the text lightly (no lowercasing, no punctuation removal)
labels = CONFIG["label_columns"]
rows_before = len(df)

df_clean = df.copy()

# 1) Strip extra whitespace (leading, trailing, repeated spaces and line breaks)
df_clean["comment_text"] = (df_clean["comment_text"].fillna("").astype(str)
                            .str.replace(r"\s+", " ", regex=True).str.strip())

# 2) Remove empty or missing text
empty = (df_clean["comment_text"] == "")
df_clean = df_clean[~empty]
print("Removed empty text:", int(empty.sum()))

# 3) Exact duplicates: first count how many duplicated texts have conflicting labels, then keep the first copy
df_clean["_flag"] = df_clean[labels].max(axis=1)
conflicts = int((df_clean.groupby("comment_text")["_flag"].nunique() > 1).sum())
dups = df_clean.duplicated(subset="comment_text", keep="first")
print("Removed exact duplicates:", int(dups.sum()), "| duplicated texts with conflicting labels:", conflicts)
df_clean = df_clean[~dups].drop(columns="_flag").reset_index(drop=True)

# Summary
print(f"\nRows before: {rows_before:,} | after: {len(df_clean):,} | removed: {rows_before - len(df_clean):,}")
flagged_share = (df_clean[labels].max(axis=1) > 0).mean()
print(f"Share of comments with at least one label after cleaning: {flagged_share:.2%}")

Removed empty text: 0
Removed exact duplicates: 197 | duplicated texts with conflicting labels: 10

Rows before: 159,571 | after: 159,374 | removed: 197
Share of comments with at least one label after cleaning: 10.16%


In [4]:
# Cell 4: create the merged label and split 70/15/15 (stratified on flagged, seed 42)
from sklearn.model_selection import train_test_split

labels = CONFIG["label_columns"]

# Merged label: 1 if ANY of the six labels is 1
df_clean["flagged"] = (df_clean[labels].max(axis=1) > 0).astype(int)

# Step 1: 70% train, 30% put aside
train_df, temp_df = train_test_split(df_clean, test_size=0.30, random_state=CONFIG["seed"],
                                     stratify=df_clean["flagged"])
# Step 2: split the 30% into two equal halves (15% validation, 15% test)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=CONFIG["seed"],
                                   stratify=temp_df["flagged"])

train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

# Sizes and share of flagged comments in each part
for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name:<11}: {len(part):>7,} comments ({len(part)/len(df_clean):.1%}) | "
          f"flagged: {part['flagged'].sum():,} ({part['flagged'].mean():.2%})")

# Save the three parts so they survive a session reset
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_csv(os.path.join(CONFIG["work_dir"], f"{name}_split.csv"), index=False)
print("\nSplits saved in", CONFIG["work_dir"], "| the test set is NOT used again until the final step")

Train      : 111,561 comments (70.0%) | flagged: 11,334 (10.16%)
Validation :  23,906 comments (15.0%) | flagged: 2,428 (10.16%)
Test       :  23,907 comments (15.0%) | flagged: 2,429 (10.16%)

Splits saved in /content/moderation | the test set is NOT used again until the final step


In [5]:
# Cell 5: baseline model, scored on the VALIDATION set only
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
import time

start = time.time()

# Turn text into word and word-pair features
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True)
X_tr = tfidf.fit_transform(train_df["comment_text"])      # fit on TRAIN only
X_va = tfidf.transform(val_df["comment_text"])

# Logistic regression with class weights (flagged comments are rare)
base_clf = LogisticRegression(class_weight="balanced", C=4, max_iter=1000, random_state=CONFIG["seed"])
base_clf.fit(X_tr, train_df["flagged"])

# Scores on the validation set (flagged class, default threshold 0.5)
base_val_prob = base_clf.predict_proba(X_va)[:, 1]
base_val_pred = (base_val_prob >= 0.5).astype(int)
y_va = val_df["flagged"].values

print("BASELINE, validation set (threshold 0.5):")
print(f"  Precision: {precision_score(y_va, base_val_pred):.3f}")
print(f"  Recall:    {recall_score(y_va, base_val_pred):.3f}")
print(f"  F1:        {f1_score(y_va, base_val_pred):.3f}")
print(f"\nFlagged in validation: {y_va.sum():,} | predicted flagged: {base_val_pred.sum():,}")
print(f"Time taken: {time.time() - start:.0f} seconds")

BASELINE, validation set (threshold 0.5):
  Precision: 0.732
  Recall:    0.843
  F1:        0.784

Flagged in validation: 2,428 | predicted flagged: 2,796
Time taken: 36 seconds


In [6]:
# Cell 6: tokenize all three parts; padding is done later, per batch (dynamic padding)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def tokenize(texts):
    # Cut long comments at max_length tokens; no padding here
    return tokenizer(list(texts), truncation=True, max_length=CONFIG["max_length"])["input_ids"]

train_ids = tokenize(train_df["comment_text"])
val_ids   = tokenize(val_df["comment_text"])
test_ids  = tokenize(test_df["comment_text"])

# How long are the tokenized comments, and how many were cut?
lens = np.array([len(x) for x in train_ids])
print("Tokenizer:", CONFIG["model_name"], "| fast tokenizer:", tokenizer.is_fast)
print("Train token length (median / 95th percentile / max):", int(np.median(lens)), "/", int(np.percentile(lens, 95)), "/", lens.max())
full_len = [len(x) for x in tokenizer(list(train_df["comment_text"].iloc[:5000]), truncation=False)["input_ids"]]
print(f"Share of comments longer than {CONFIG['max_length']} tokens (first 5,000 train rows): {np.mean(np.array(full_len) > CONFIG['max_length']):.1%}")
print("Example:", train_df["comment_text"].iloc[0][:80], "->", train_ids[0][:12], "...")
print("Sizes -> train:", len(train_ids), "| validation:", len(val_ids), "| test:", len(test_ids))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer: distilbert-base-uncased | fast tokenizer: True
Train token length (median / 95th percentile / max): 52 / 128 / 128


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1370 > 512). Running this sequence through the model will result in indexing errors


Share of comments longer than 128 tokens (first 5,000 train rows): 20.2%
Example: " Here on Wikipedia, my daughter was treated with insolence, which is unacceptab -> [101, 1000, 2182, 2006, 16948, 1010, 2026, 2684, 2001, 5845, 2007, 16021] ...
Sizes -> train: 111561 | validation: 23906 | test: 23907


In [7]:
# Cell 7: fine-tune DistilBERT (class-weighted loss, fp16), keep the best epoch by validation F1
import time, math
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import precision_score, recall_score, f1_score

device = "cuda"
pad_id = tokenizer.pad_token_id
y_train = train_df["flagged"].values
y_val = val_df["flagged"].values

# Build one padded batch (padding only to the longest comment in the batch)
def make_batch(ids_list, idx):
    seqs = [ids_list[i] for i in idx]
    m = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), m), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), m), dtype=torch.long)
    for j, s in enumerate(seqs):
        input_ids[j, :len(s)] = torch.tensor(s)
        mask[j, :len(s)] = 1
    return input_ids.to(device), mask.to(device)

# Probability of "flagged" for every comment (sorted by length for speed, returned in original order)
def predict_probs(model, ids_list, batch_size=128):
    model.eval()
    order = np.argsort([len(x) for x in ids_list])
    probs = np.zeros(len(ids_list))
    with torch.no_grad():
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            inp, mask = make_batch(ids_list, idx)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=inp, attention_mask=mask).logits
            probs[idx] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    return probs

# Model, class weights (rare flagged class gets a bigger weight), optimizer, learning-rate schedule
model = AutoModelForSequenceClassification.from_pretrained(CONFIG["model_name"], num_labels=2).to(device)
n_pos = int(y_train.sum()); n_neg = len(y_train) - n_pos
weights = torch.tensor([len(y_train) / (2 * n_neg), len(y_train) / (2 * n_pos)], dtype=torch.float).to(device)
print("Class weights (clean, flagged):", [round(w, 3) for w in weights.tolist()])
loss_fn = nn.CrossEntropyLoss(weight=weights)

steps_per_epoch = math.ceil(len(train_ids) / CONFIG["batch_size"])
total_steps = steps_per_epoch * CONFIG["epochs"]
warmup = int(0.06 * total_steps)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: s / max(1, warmup) if s < warmup else max(0.0, (total_steps - s) / max(1, total_steps - warmup)))
scaler = torch.amp.GradScaler("cuda")

best_dir = os.path.join(CONFIG["work_dir"], "best_model")
rng = np.random.default_rng(CONFIG["seed"])
best_f1, history, step = -1.0, [], 0
t_start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    perm = rng.permutation(len(train_ids))
    running = 0.0
    for b in range(steps_per_epoch):
        idx = perm[b * CONFIG["batch_size"]:(b + 1) * CONFIG["batch_size"]]
        inp, mask = make_batch(train_ids, idx)
        target = torch.tensor(y_train[idx], dtype=torch.long).to(device)
        optimizer.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=inp, attention_mask=mask).logits
        loss = loss_fn(logits.float(), target)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item(); step += 1
        if step % 500 == 0:
            print(f"  epoch {epoch} | step {step}/{total_steps} | avg loss {running / 500:.4f} | {(time.time() - t_start) / 60:.1f} min")
            running = 0.0

    # Evaluate on the VALIDATION set (threshold 0.5) after every epoch
    val_prob = predict_probs(model, val_ids)
    pred = (val_prob >= 0.5).astype(int)
    p, r, f = precision_score(y_val, pred), recall_score(y_val, pred), f1_score(y_val, pred)
    history.append({"epoch": epoch, "precision": p, "recall": r, "f1": f})
    print(f"Epoch {epoch} validation -> precision {p:.3f} | recall {r:.3f} | F1 {f:.3f}")

    # Keep the best epoch on disk (model, tokenizer and its validation probabilities)
    if f > best_f1:
        best_f1, best_epoch = f, epoch
        model.save_pretrained(best_dir); tokenizer.save_pretrained(best_dir)
        np.save(os.path.join(CONFIG["work_dir"], "val_prob_distilbert.npy"), val_prob)
        print("  -> best so far, saved to", best_dir)

print(f"\nTraining finished in {(time.time() - t_start) / 60:.1f} minutes | best epoch: {best_epoch} | best validation F1: {best_f1:.3f}")
print("Baseline validation F1 for comparison: 0.784")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights (clean, flagged): [0.557, 4.922]
  epoch 1 | step 500/6974 | avg loss 0.3754 | 0.8 min
  epoch 1 | step 1000/6974 | avg loss 0.2423 | 1.7 min
  epoch 1 | step 1500/6974 | avg loss 0.2235 | 2.6 min
  epoch 1 | step 2000/6974 | avg loss 0.2145 | 3.4 min
  epoch 1 | step 2500/6974 | avg loss 0.2082 | 4.3 min
  epoch 1 | step 3000/6974 | avg loss 0.2044 | 5.1 min
Epoch 1 validation -> precision 0.807 | recall 0.875 | F1 0.840


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> best so far, saved to /content/moderation/best_model
  epoch 2 | step 3500/6974 | avg loss 0.0025 | 6.2 min
  epoch 2 | step 4000/6974 | avg loss 0.1513 | 7.1 min
  epoch 2 | step 4500/6974 | avg loss 0.1528 | 7.9 min
  epoch 2 | step 5000/6974 | avg loss 0.1431 | 8.7 min
  epoch 2 | step 5500/6974 | avg loss 0.1568 | 9.6 min
  epoch 2 | step 6000/6974 | avg loss 0.1533 | 10.4 min
  epoch 2 | step 6500/6974 | avg loss 0.1378 | 11.2 min
Epoch 2 validation -> precision 0.805 | recall 0.880 | F1 0.841


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> best so far, saved to /content/moderation/best_model

Training finished in 12.2 minutes | best epoch: 2 | best validation F1: 0.841
Baseline validation F1 for comparison: 0.784


In [9]:
# Cell 8: score validation set with best model, try thresholds 0.1 to 0.9
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_score, recall_score, f1_score

best_dir = "/content/moderation/best_model"
tok = AutoTokenizer.from_pretrained(best_dir)
mdl = AutoModelForSequenceClassification.from_pretrained(best_dir).to("cuda").eval()

# Get probability of "flagged" for every validation text
texts = list(val_df["comment_text"])
y_val = np.array(val_df["flagged"])
scores = []
with torch.no_grad():
    for i in range(0, len(texts), 128):
        enc = tok(texts[i:i+128], truncation=True, max_length=128,
                  padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = mdl(**enc).logits
        scores.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
scores = np.array(scores)
np.save("/content/moderation/val_scores.npy", scores)

# Table of thresholds
rows = []
for t in np.arange(0.1, 0.91, 0.1):
    pred = (scores >= t).astype(int)
    rows.append({"threshold": round(t, 1),
                 "precision": round(precision_score(y_val, pred), 3),
                 "recall": round(recall_score(y_val, pred), 3),
                 "F1": round(f1_score(y_val, pred), 3)})
print(pd.DataFrame(rows).to_string(index=False))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

 threshold  precision  recall    F1
       0.1      0.692   0.930 0.794
       0.2      0.745   0.911 0.820
       0.3      0.771   0.895 0.829
       0.4      0.789   0.887 0.835
       0.5      0.805   0.880 0.840
       0.6      0.817   0.870 0.843
       0.7      0.829   0.864 0.846
       0.8      0.846   0.850 0.848
       0.9      0.871   0.830 0.850


In [10]:
# Cell 9: FINAL test evaluation at threshold 0.5 (run ONCE, do not tune after this)
import numpy as np, pandas as pd, torch
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score, confusion_matrix)

THRESHOLD = 0.5
labels = CONFIG["label_columns"]
y_te = test_df["flagged"].values

# DistilBERT scores on the test set (reuses tok and mdl from cell 8)
texts = list(test_df["comment_text"])
bert_prob = []
with torch.no_grad():
    for i in range(0, len(texts), 128):
        enc = tok(texts[i:i+128], truncation=True, max_length=128,
                  padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = mdl(**enc).logits
        bert_prob.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
bert_prob = np.array(bert_prob)

# Baseline scores on the test set (reuses tfidf and base_clf from cell 5)
base_prob = base_clf.predict_proba(tfidf.transform(test_df["comment_text"]))[:, 1]
np.save("/content/moderation/test_scores_bert.npy", bert_prob)
np.save("/content/moderation/test_scores_base.npy", base_prob)

# Main metrics for the flagged class
for name, prob in [("Baseline (TF-IDF + LR)", base_prob), ("DistilBERT", bert_prob)]:
    pred = (prob >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    print(f"\n{name} | threshold {THRESHOLD}")
    print(f"  Precision: {precision_score(y_te, pred):.3f}")
    print(f"  Recall:    {recall_score(y_te, pred):.3f}")
    print(f"  F1:        {f1_score(y_te, pred):.3f}")
    print(f"  PR-AUC:    {average_precision_score(y_te, prob):.3f}")
    print(f"  Confusion matrix -> clean OK: {tn:,} | false flags: {fp:,} | missed: {fn:,} | caught: {tp:,}")

# Recall for each of the six labels separately
rows = []
for lab in labels:
    m = test_df[lab].values == 1
    rows.append({"label": lab, "n_in_test": int(m.sum()),
                 "baseline_recall": round(float((base_prob[m] >= THRESHOLD).mean()), 3),
                 "distilbert_recall": round(float((bert_prob[m] >= THRESHOLD).mean()), 3)})
print("\nRecall per label (share of comments with that label that were flagged):")
print(pd.DataFrame(rows).to_string(index=False))
print("\nThreat comments in the test set:", int(test_df["threat"].sum()))


Baseline (TF-IDF + LR) | threshold 0.5
  Precision: 0.734
  Recall:    0.810
  F1:        0.770
  PR-AUC:    0.870
  Confusion matrix -> clean OK: 20,764 | false flags: 714 | missed: 462 | caught: 1,967

DistilBERT | threshold 0.5
  Precision: 0.807
  Recall:    0.862
  F1:        0.833
  PR-AUC:    0.921
  Confusion matrix -> clean OK: 20,977 | false flags: 501 | missed: 336 | caught: 2,093

Recall per label (share of comments with that label that were flagged):
        label  n_in_test  baseline_recall  distilbert_recall
        toxic       2278            0.823              0.873
 severe_toxic        222            0.982              0.995
      obscene       1253            0.930              0.966
       threat         76            0.895              0.934
       insult       1157            0.927              0.969
identity_hate        191            0.874              0.953

Threat comments in the test set: 76


In [11]:
# Cell 10: hand-made ride-chat messages (label 1 = should be flagged, 0 = normal)
import pandas as pd
normal = [
 "where are you", "I am at the gate", "please don't be late", "thanks bhai",
 "this traffic is killing me", "I'm dying of heat, can you turn on the AC?",
 "you drive crazy good", "I will be there in 5 minutes", "which gate should I come to?",
 "stop near the petrol pump please", "hurry up, I am late for my train",
 "what the hell is this traffic", "damn it's hot today", "I am wearing a red jacket",
 "can you wait two minutes?", "sorry, my phone was dead", "turn left at the next signal",
 "please drive slowly, I feel sick", "you missed the turn, let's take the next one",
 "I booked this ride for two people", "ok see you soon", "bhaiya thoda jaldi aao",
 "kitna time lagega?", "I'll pay by cash, is that fine?", "this road is a nightmare",
 "my bag is heavy, can you help?", "great ride, thank you!", "you are so slow today lol",
 "I am so angry at the traffic", "kill the music a bit please",
]
abusive = [
 "you are a stupid idiot", "shut up you moron", "you are useless, get lost",
 "I will find you and hurt you", "I will kill you if you cancel", "I know where you live",
 "you will regret this, I am coming for you", "I'll break your legs", "you filthy pig",
 "go to hell you jerk", "worst driver ever, you dumb fool", "I hate you, you piece of trash",
 "you better watch your back", "women like you should stay at home",
 "I will make sure you never drive again or else", "you ugly loser",
 "tujhe jaan se maar dunga", "tere haath tod dunga", "saale kutte, kahan hai tu",
 "bewakoof insaan, chup kar", "tu pagal hai kya, gadha", "tujhe dekh lunga",
 "nikal yahan se kamine", "tujhe chhodunga nahi", "kya bakwas driver hai, nalayak",
 "I'll beat you up when you arrive", "you disgusting creature", "idiot, are you blind?",
 "I will burn your car", "you are a worthless bastard",
]
san = pd.DataFrame({"text": normal + abusive, "label": [0]*len(normal) + [1]*len(abusive)})
san.to_csv("/content/moderation/sanity_messages_draft.csv", index=False)

print(f"Total: {len(san)} | normal (0): {len(normal)} | abusive/threat (1): {len(abusive)}\n")
for i, r in san.iterrows():
    print(f"{i:>2} | label {r['label']} | {r['text']}")

Total: 60 | normal (0): 30 | abusive/threat (1): 30

 0 | label 0 | where are you
 1 | label 0 | I am at the gate
 2 | label 0 | please don't be late
 3 | label 0 | thanks bhai
 4 | label 0 | this traffic is killing me
 5 | label 0 | I'm dying of heat, can you turn on the AC?
 6 | label 0 | you drive crazy good
 7 | label 0 | I will be there in 5 minutes
 8 | label 0 | which gate should I come to?
 9 | label 0 | stop near the petrol pump please
10 | label 0 | hurry up, I am late for my train
11 | label 0 | what the hell is this traffic
12 | label 0 | damn it's hot today
13 | label 0 | I am wearing a red jacket
14 | label 0 | can you wait two minutes?
15 | label 0 | sorry, my phone was dead
16 | label 0 | turn left at the next signal
17 | label 0 | please drive slowly, I feel sick
18 | label 0 | you missed the turn, let's take the next one
19 | label 0 | I booked this ride for two people
20 | label 0 | ok see you soon
21 | label 0 | bhaiya thoda jaldi aao
22 | label 0 | kitna time lageg

In [12]:
# Cell 11: run DistilBERT (threshold 0.5) on the 60 hand-made ride-chat messages
import torch, numpy as np, pandas as pd

THRESHOLD = 0.5
san = pd.read_csv("/content/moderation/sanity_messages_draft.csv")
san.to_csv("/content/moderation/sanity_messages.csv", index=False)   # final labelled copy

# Score every message (reuses tok and mdl from cell 8)
scores = []
with torch.no_grad():
    for i in range(0, len(san), 32):
        enc = tok(list(san["text"][i:i+32]), truncation=True, max_length=128,
                  padding=True, return_tensors="pt").to("cuda")
        logits = mdl(**enc).logits
        scores.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
san["score"] = np.round(scores, 3)
san["pred"] = (san["score"] >= THRESHOLD).astype(int)
san.to_csv("/content/moderation/sanity_results.csv", index=False)

# False flags (normal but flagged) and missed (abusive but not flagged)
false_flags = san[(san["label"] == 0) & (san["pred"] == 1)]
missed = san[(san["label"] == 1) & (san["pred"] == 0)]
print("FALSE FLAGS (normal messages that were flagged):")
print(false_flags[["text", "score"]].to_string() if len(false_flags) else "  none")
print("\nMISSED (abusive messages that were NOT flagged):")
print(missed[["text", "score"]].to_string() if len(missed) else "  none")

# Overall counts, plus Hindi-in-English-letters messages (items 46 to 54) shown separately
print(f"\nNormal correctly passed: {((san.label==0)&(san.pred==0)).sum()} / {(san.label==0).sum()}")
print(f"Abusive correctly flagged: {((san.label==1)&(san.pred==1)).sum()} / {(san.label==1).sum()}")
hindi = san.iloc[46:55]
print(f"Hindi-style abusive (46-54) flagged: {int(hindi['pred'].sum())} / {len(hindi)}")

FALSE FLAGS (normal messages that were flagged):
                             text  score
4      this traffic is killing me  0.853
6            you drive crazy good  0.779
11  what the hell is this traffic  0.987
12            damn it's hot today  0.914
24       this road is a nightmare  0.831
27      you are so slow today lol  0.937

MISSED (abusive messages that were NOT flagged):
                                              text  score
35                           I know where you live  0.002
36       you will regret this, I am coming for you  0.011
42                      you better watch your back  0.462
43              women like you should stay at home  0.326
44  I will make sure you never drive again or else  0.014
46                        tujhe jaan se maar dunga  0.214
47                            tere haath tod dunga  0.364
48                       saale kutte, kahan hai tu  0.081
49                       bewakoof insaan, chup kar  0.021
50                         tu paga

In [13]:
# Cell 12: model size on disk, and time per message on CPU and GPU
import os, time, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

best_dir = "/content/moderation/best_model"

# Size of the saved model folder in MB
total = 0
for f in os.listdir(best_dir):
    s = os.path.getsize(os.path.join(best_dir, f)) / 1e6
    total += s
    print(f"  {f:<28} {s:8.1f} MB")
print(f"Model folder size: {total:.1f} MB\n")

# 120 real messages (the 60 sanity messages twice), scored ONE at a time like the real service
msgs = list(pd.read_csv("/content/moderation/sanity_messages.csv")["text"]) * 2
tok_s = AutoTokenizer.from_pretrained(best_dir)

def time_model(model, device, n_warmup=5):
    times = []
    with torch.no_grad():
        for i, m in enumerate(msgs + msgs[:n_warmup]):
            t0 = time.perf_counter()
            enc = tok_s(m, truncation=True, max_length=128, return_tensors="pt").to(device)
            _ = torch.softmax(model(**enc).logits, dim=-1)
            if device == "cuda":
                torch.cuda.synchronize()
            dt = (time.perf_counter() - t0) * 1000
            if i >= n_warmup:               # ignore the first few warm-up calls
                times.append(dt)
    return np.mean(times), np.percentile(times, 95)

# CPU (forced, float32)
cpu_model = AutoModelForSequenceClassification.from_pretrained(best_dir).to("cpu").eval()
cpu_avg, cpu_p95 = time_model(cpu_model, "cpu")
print(f"CPU threads available: {torch.get_num_threads()}")
print(f"CPU time per message: average {cpu_avg:.1f} ms | 95th percentile {cpu_p95:.1f} ms")

# GPU
gpu_model = AutoModelForSequenceClassification.from_pretrained(best_dir).to("cuda").eval()
gpu_avg, gpu_p95 = time_model(gpu_model, "cuda")
print(f"GPU time per message: average {gpu_avg:.1f} ms | 95th percentile {gpu_p95:.1f} ms")

# Check against the limits from the plan
print("\nCPU time above 300 ms?", "YES" if cpu_avg > 300 else "no", "| model above ~300 MB?", "YES" if total > 300 else "no")

  tokenizer_config.json             0.0 MB
  model.safetensors               267.8 MB
  tokenizer.json                    0.7 MB
  config.json                       0.0 MB
Model folder size: 268.5 MB



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

CPU threads available: 1
CPU time per message: average 90.3 ms | 95th percentile 214.4 ms


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

GPU time per message: average 9.7 ms | 95th percentile 17.5 ms

CPU time above 300 ms? no | model above ~300 MB? no


In [14]:
# Cell 13: save deployable files in a NEW folder (nothing is overwritten) and verify them
import os, json, time, datetime, shutil, numpy as np, pandas as pd, torch, sklearn, transformers
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

THRESHOLD = 0.5
labels = CONFIG["label_columns"]

# New folder; if the name already exists, add a time stamp instead of overwriting
out = "/content/moderation/deploy_v1"
if os.path.exists(out):
    out = out + "_" + time.strftime("%Y%m%d_%H%M%S")
os.makedirs(out)

# 1) Model and tokenizer (CPU float32 copy from cell 12)
cpu_model.save_pretrained(os.path.join(out, "model"))
tok_s.save_pretrained(os.path.join(out, "model"))

# 2) moderate(text) code saved as a file for the Python service
moderate_code = '''import json, os, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def load_moderator(folder):
    cfg = json.load(open(os.path.join(folder, "moderation_config.json")))
    mdir = os.path.join(folder, "model")
    tok = AutoTokenizer.from_pretrained(mdir)
    model = AutoModelForSequenceClassification.from_pretrained(mdir).to("cpu").eval()
    thr, max_len = cfg["threshold"], cfg["max_length"]
    def moderate(text):
        # empty or non-text input is never flagged
        if not isinstance(text, str) or not text.strip():
            return {"flagged": False, "score": 0.0, "threshold": thr}
        text = " ".join(text.split())[:2000]   # clean spaces, cut very long text
        enc = tok(text, truncation=True, max_length=max_len, return_tensors="pt")
        with torch.no_grad():
            p = torch.softmax(model(**enc).logits, dim=-1)[0, 1].item()
        return {"flagged": bool(p >= thr), "score": round(p, 4), "threshold": thr}
    return moderate
'''
open(os.path.join(out, "moderate.py"), "w").write(moderate_code)

# 3) Config file
config = {
    "threshold": THRESHOLD, "max_length": 128, "model_name": CONFIG["model_name"],
    "label_merge_rule": "flagged = 1 if any of toxic, severe_toxic, obscene, threat, insult, identity_hate is 1, else 0",
    "training_data": "Kaggle Jigsaw Toxic Comment Classification Challenge, train.csv only (Wikipedia comments, English). 70/15/15 stratified split, seed 42.",
    "epochs": 2, "best_epoch": 2,
    "library_versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                         "scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "training_date": datetime.date.today().isoformat(),
}
json.dump(config, open(os.path.join(out, "moderation_config.json"), "w"), indent=2)

# 4) Sanity-check files
shutil.copy("/content/moderation/sanity_messages.csv", os.path.join(out, "sanity_messages.csv"))
shutil.copy("/content/moderation/sanity_results.csv", os.path.join(out, "sanity_results.csv"))

# 5) Metrics file (test scores were saved in cell 9, sanity results in cell 11)
y_te = test_df["flagged"].values
metrics = {"threshold": THRESHOLD,
           "dataset": {"rows_raw": len(df), "rows_after_cleaning": len(df_clean),
                       "train": len(train_df), "validation": len(val_df), "test": len(test_df),
                       "flagged_share_after_cleaning": round(float(df_clean["flagged"].mean()), 4)}}
for name, f in [("baseline", "test_scores_base.npy"), ("distilbert", "test_scores_bert.npy")]:
    prob = np.load("/content/moderation/" + f); pred = (prob >= THRESHOLD).astype(int)
    tn, fp, fn, tp = [int(x) for x in confusion_matrix(y_te, pred).ravel()]
    metrics[name + "_test"] = {"precision": round(precision_score(y_te, pred), 3),
        "recall": round(recall_score(y_te, pred), 3), "f1": round(f1_score(y_te, pred), 3),
        "pr_auc": round(average_precision_score(y_te, prob), 3),
        "confusion": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
        "per_label_recall": {l: round(float((prob[test_df[l].values == 1] >= THRESHOLD).mean()), 3) for l in labels}}
sr = pd.read_csv("/content/moderation/sanity_results.csv")
metrics["sanity_check"] = {"normal_passed": int(((sr.label == 0) & (sr.pred == 0)).sum()), "normal_total": int((sr.label == 0).sum()),
                           "abusive_flagged": int(((sr.label == 1) & (sr.pred == 1)).sum()), "abusive_total": int((sr.label == 1).sum())}
metrics["speed_size"] = {"model_mb": round(total, 1), "cpu_ms_avg": round(cpu_avg, 1), "cpu_ms_p95": round(cpu_p95, 1),
                         "gpu_ms_avg": round(gpu_avg, 1), "gpu_ms_p95": round(gpu_p95, 1), "cpu_threads_used": torch.get_num_threads()}
json.dump(metrics, open(os.path.join(out, "moderation_metrics.json"), "w"), indent=2)

# 6) Golden test cases: expected scores from the in-memory CPU model
golden_idx = [0, 1, 3, 4, 7, 16, 21, 26, 30, 31, 33, 34, 37, 41, 55]
golden_texts = [sr["text"].iloc[i] for i in golden_idx]
def raw_score(t):
    enc = tok_s(t, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        return torch.softmax(cpu_model(**enc).logits, dim=-1)[0, 1].item()
golden = [{"text": t, "expected_score": round(raw_score(t), 4),
           "expected_flagged": bool(raw_score(t) >= THRESHOLD)} for t in golden_texts]
json.dump({"tolerance": 0.001, "cases": golden}, open(os.path.join(out, "golden_test_cases.json"), "w"), indent=2)

# 7) Reload EVERYTHING from disk and check the golden cases
import importlib.util
spec = importlib.util.spec_from_file_location("moderate_mod", os.path.join(out, "moderate.py"))
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
moderate = mod.load_moderator(out)

diffs = [abs(moderate(g["text"])["score"] - g["expected_score"]) for g in golden]
flags_ok = all(moderate(g["text"])["flagged"] == g["expected_flagged"] for g in golden)
print("Saved in:", out)
for root, _, files in os.walk(out):
    for f in files:
        print("  ", os.path.relpath(os.path.join(root, f), out))
print(f"\nGolden cases: {len(golden)} | largest score difference: {max(diffs):.5f} | all flag decisions match: {flags_ok}")
print("Empty text ->", moderate(""))
print("Only spaces ->", moderate("   "))
print("None ->", moderate(None))
print("Very long text ->", moderate("you are an idiot " * 2000))
print("Normal example ->", moderate("where are you"))
print("\nRELOAD CHECK:", "PASSED" if max(diffs) <= 0.001 and flags_ok else "FAILED")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Saved in: /content/moderation/deploy_v1
   sanity_results.csv
   sanity_messages.csv
   moderation_config.json
   moderate.py
   moderation_metrics.json
   golden_test_cases.json
   model/tokenizer_config.json
   model/model.safetensors
   model/tokenizer.json
   model/config.json
   __pycache__/moderate.cpython-313.pyc

Golden cases: 15 | largest score difference: 0.00000 | all flag decisions match: True
Empty text -> {'flagged': False, 'score': 0.0, 'threshold': 0.5}
Only spaces -> {'flagged': False, 'score': 0.0, 'threshold': 0.5}
None -> {'flagged': False, 'score': 0.0, 'threshold': 0.5}
Very long text -> {'flagged': True, 'score': 0.9996, 'threshold': 0.5}
Normal example -> {'flagged': False, 'score': 0.0086, 'threshold': 0.5}

RELOAD CHECK: PASSED


In [15]:
# Cell 14: final summary of numbers for the report, read from the saved files, plus a zip of the deploy folder
import json, os, shutil
from google.colab import files

out = "/content/moderation/deploy_v1"      # change only if cell 13 printed a different folder name
m = json.load(open(os.path.join(out, "moderation_metrics.json")))
c = json.load(open(os.path.join(out, "moderation_config.json")))
d = m["dataset"]

print("=== NUMBERS FOR MY REPORT ===\n")
print("DATASET")
print(f"  Raw rows: {d['rows_raw']:,} | after cleaning: {d['rows_after_cleaning']:,} | removed: {d['rows_raw'] - d['rows_after_cleaning']:,}")
print(f"  Train: {d['train']:,} | Validation: {d['validation']:,} | Test: {d['test']:,}")
print(f"  Share flagged (after cleaning): {d['flagged_share_after_cleaning']:.2%}")
print(f"\nTRAINING: epochs {c['epochs']} (best epoch {c['best_epoch']}) | threshold {m['threshold']} | max_length {c['max_length']}")

print("\nTEST SET (flagged class, threshold", m["threshold"], ")")
print(f"  {'Model':<12}{'Precision':>10}{'Recall':>9}{'F1':>8}{'PR-AUC':>9}")
for k, name in [("baseline_test", "Baseline"), ("distilbert_test", "DistilBERT")]:
    r = m[k]
    print(f"  {name:<12}{r['precision']:>10}{r['recall']:>9}{r['f1']:>8}{r['pr_auc']:>9}")
for k, name in [("baseline_test", "Baseline"), ("distilbert_test", "DistilBERT")]:
    cm = m[k]["confusion"]
    print(f"  {name} confusion -> clean OK {cm['tn']:,} | false flags {cm['fp']:,} | missed {cm['fn']:,} | caught {cm['tp']:,}")

print("\nPER-LABEL RECALL (test set)")
print(f"  {'Label':<15}{'Baseline':>10}{'DistilBERT':>12}")
for lab in m["distilbert_test"]["per_label_recall"]:
    print(f"  {lab:<15}{m['baseline_test']['per_label_recall'][lab]:>10}{m['distilbert_test']['per_label_recall'][lab]:>12}")
print("  (threat comments in test set: 76)")

s = m["sanity_check"]; sp = m["speed_size"]
print("\nRIDE-CHAT SANITY CHECK (60 hand-made messages, not a benchmark)")
print(f"  Normal passed: {s['normal_passed']}/{s['normal_total']} | abusive flagged: {s['abusive_flagged']}/{s['abusive_total']}")
print("  Hindi in English letters flagged: 0/9 (training data is English only)")

print("\nSIZE AND SPEED")
print(f"  Model size: {sp['model_mb']} MB")
print(f"  CPU ({sp['cpu_threads_used']} thread): average {sp['cpu_ms_avg']} ms | 95th percentile {sp['cpu_ms_p95']} ms per message")
print(f"  GPU: average {sp['gpu_ms_avg']} ms | 95th percentile {sp['gpu_ms_p95']} ms per message")

# Zip the deploy folder for download
zip_path = shutil.make_archive("/content/QuickPull_moderation_deploy", "zip", out)
print("\nZip created:", zip_path, f"({os.path.getsize(zip_path)/1e6:.0f} MB)")
files.download(zip_path)

=== NUMBERS FOR MY REPORT ===

DATASET
  Raw rows: 159,571 | after cleaning: 159,374 | removed: 197
  Train: 111,561 | Validation: 23,906 | Test: 23,907
  Share flagged (after cleaning): 10.16%

TRAINING: epochs 2 (best epoch 2) | threshold 0.5 | max_length 128

TEST SET (flagged class, threshold 0.5 )
  Model        Precision   Recall      F1   PR-AUC
  Baseline         0.734     0.81    0.77     0.87
  DistilBERT       0.807    0.862   0.833    0.921
  Baseline confusion -> clean OK 20,764 | false flags 714 | missed 462 | caught 1,967
  DistilBERT confusion -> clean OK 20,977 | false flags 501 | missed 336 | caught 2,093

PER-LABEL RECALL (test set)
  Label            Baseline  DistilBERT
  toxic               0.823       0.873
  severe_toxic        0.982       0.995
  obscene              0.93       0.966
  threat              0.895       0.934
  insult              0.927       0.969
  identity_hate       0.874       0.953
  (threat comments in test set: 76)

RIDE-CHAT SANITY CHECK 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>